# 🧬 GroupDNA — Your WhatsApp Group Chat, Decoded

---

| Field | Details |
|---|---|
| **Project** | GroupDNA — Your WhatsApp Group Chat, Decoded |
| **Name** | *Anjali Ajay Chavan* |
| **Date** | June 2026 |
| **Dataset** | `hostel_bois.txt` — 60 days, 6 participants, ~3,174 messages |

---

## 🎯 Objective

GroupDNA is a behavioural analytics engine built on a raw WhatsApp group chat export. The goal is to parse thousands of timestamped messages, uncover hidden patterns in how people communicate, and assign a personality archetype to every group member — all without using a single external data library.

###Feature 1 - The Chat Parse

In [ ]:
from datetime import datetime, timedelta
import numpy as np
import string

#Load the file. (Using-8 encoding)
with open('hostel_bois.txt', 'r', encoding='utf-8') as chat_file:
  raw_content = chat_file.read()
raw_lines = raw_content.split('\n')

#Storing every parsed message as a dict
#'str':= timestamp, sender, text
#'type':= real/deleted/media
#'dt':= datetime
#message (real + media + deleted)
Messages = []
Summary = {
    'sys_msg' : 0,
    'media' : 0,
    'deleted' : 0,
    'empty_lines' : 0,
}
#Track the index of the last parsed message. (needed for handling multi line messages)
last_real_idx = -1

def Date_Format(ln):
  return len(ln)>8 and ln[2] == '/' and ln[5] =='/'

#Used AI-Assistant here for parse_timestamp
def parse_timestamp(ts):
  try:
    return datetime.strptime(ts, '%d/%m/%y, %H:%M')
  except ValueError:
      return None

for raw_line in raw_lines:
  line = raw_line.strip()

  #Empty lines:No data-> Skip
  if len(line) ==0:
    Summary['empty_lines'] += 1 # Corrected: 'line' to 'Summary'
    continue

  #Multi line message
  if not Date_Format(line):
    if last_real_idx >= 0:
        Messages[last_real_idx]['Message_Text'] += ' ' + line
    continue

  #Spliting
  split_dash = line.split('-', 1)
  if len(split_dash) < 2:
    Summary['sys_msg'] += 1
    continue
  timestamp_part = split_dash[0].strip()
  msg_line = split_dash[1].strip()

  parsed_dt= parse_timestamp(timestamp_part)
  if parsed_dt is None:
    Summary['sys_msg'] += 1
    continue

  #System message check (if ':' is not in the message line, it's a system message -> skip)
  if ': ' not in msg_line:
    Summary['sys_msg'] += 1
    continue

  #Split sender + message
  sender_name, text = msg_line.split(':', 1)

  #Identify message type
  if '<Media omitted>' in text:
    msg_type = 'media'
    Summary['media'] += 1
  elif 'This message was deleted' in text:
    msg_type = 'deleted'
    Summary['deleted'] += 1
  else:
    msg_type = 'real'

  record = {
      'Timestamp': timestamp_part,
      'Sender': sender_name,
      'Message_Text': text,
      'Type': msg_type,
      'Datetime': parsed_dt
  }
  Messages.append(record)
  last_real_idx = len(Messages) -1

# Corrected: Functions moved out of the loop and properly indented
# Participants
def get_participants(messages_list):
  participants= set()
  senders= []
  for msg in messages_list:
    sender= msg['Sender']
    if sender not in participants:
      participants.add(sender)
      senders.append(sender)
  return senders

# Date range (Used AI-assistant here )
def get_date_range(messages_list):
  all_dates  = sorted(set(msg['Datetime'].date() for msg in messages_list))
  start_date = all_dates[0]
  end_date   = all_dates[-1]
  total_days = (end_date - start_date).days + 1
  return start_date, end_date, total_days

# Separte message types
real_messages, media_messages, deleted_messages = [], [], []

for msg in Messages: # Corrected: 'messages' to 'Messages'
  if msg['Type']=='real':
    real_messages.append(msg)
  elif msg['Type']=='media':
    media_messages.append(msg)
  elif msg['Type']=='deleted':
    deleted_messages.append(msg)

member_list = get_participants(Messages) # Pass the full Messages list to get all participants
start_date, end_date, total_days = get_date_range(real_messages)

# Print summary( Used AI- assistant here for chat parse-summary)
print("=" * 55)
print("         CHAT PARSER — SUMMARY")
print("=" * 55)
print(f"  Total real messages  : {len(real_messages):,}")
print(f"  Total participants   : {len(member_list)}")
print(f"  Chat duration        : {total_days} days")
print(f"  From                 : {start_date.strftime('%d %B %Y')}")
print(f"  To                   : {end_date.strftime('%d %B %Y')}")
print()
print(f"  System messages skipped  : {Summary['sys_msg']}")
print(f"  Media messages counted   : {Summary['media']}")
print(f"  Deleted messages counted : {Summary['deleted']}")
print("=" * 55)
print()
print("  First 3 messages:")
for msg in real_messages[:3]:
    print(f"    [{msg['Timestamp']}] {msg['Sender']}: {msg['Message_Text'][:50]}")
print()
print("  Last 3 messages:")
for msg in real_messages[-3:]:
    print(f"    [{msg['Timestamp']}] {msg['Sender']}: {msg['Message_Text'][:50]}")
print("=" * 55)

         CHAT PARSER — SUMMARY
  Total real messages  : 3,127
  Total participants   : 6
  Chat duration        : 60 days
  From                 : 01 April 2024
  To                   : 30 May 2024

  System messages skipped  : 4
  Media messages counted   : 32
  Deleted messages counted : 15

  First 3 messages:
    [01/04/24, 01:17] Rahul:  scene fix
    [01/04/24, 01:17] Rahul:  haan
    [01/04/24, 01:18] Rahul:  kya scene

  Last 3 messages:
    [30/05/24, 21:17] Aman:  the existential dread is back
    [30/05/24, 21:30] Karan:  Long day guys, woke up at six for that placement 
    [30/05/24, 23:31] Aman:  anyone awake?


###Feature 2 - The Group Review

The executive summary of the report. Total messages, date range, and a ranked leaderboard of message counts with an inline bar chart (made of block characters, not matplotlib).

In [ ]:
# Count messages per participant
msg_count_per_person = {}

for msg in real_messages:
    sender = msg['Sender']
    if sender in msg_count_per_person:
        msg_count_per_person[sender] += 1
    else:
        msg_count_per_person[sender] = 1

# Finding total no of real messages
total_msg_count = 0
for count in msg_count_per_person.values():
    total_msg_count += count

# Sorting participants accordingly to message count (Highest -> Lowest)
ranked_senders = sorted(msg_count_per_person.items(), reverse=True, key=lambda x: x[1])

# Counting words per person
word_count_per_person = {}
for msg in real_messages:
    sender = msg['Sender']
    word_count_per_person[sender] = word_count_per_person.get(sender, 0) + len(msg['Message_Text'].split())

# Average words per participant #used ai-assistant here
avg_words = {}
for sender in member_list:
    total_words = word_count_per_person.get(sender, 0)
    person_msg  = msg_count_per_person.get(sender, 0)
    if person_msg > 0:
        avg_words[sender] = round(total_words / person_msg, 1)
    else:
        avg_words[sender] = 0.0

# Count media and deleted per participant
media_count   = {}
deleted_count = {}

for msg in media_messages:
    sender = msg['Sender']
    if sender in media_count:
        media_count[sender] += 1
    else:
        media_count[sender] = 1

for msg in deleted_messages:
    sender = msg['Sender']
    if sender in deleted_count:
        deleted_count[sender] += 1
    else:
        deleted_count[sender] = 1

#used AI-assistant here
# Bar chart function
def make_bar(value, max_val):
    bar_length = 20
    if max_val == 0:
        return '.' * bar_length
    else:
        filled_boxes = int(round((value / max_val) * bar_length))
        empty_boxes  = bar_length - filled_boxes
        return '█' * filled_boxes + '.' * empty_boxes

# Report printing
#used Ai-assistant here
highest_count = ranked_senders[0][1] if ranked_senders else 1

print('=' * 60)
print('         GROUP OVERVIEW — HOSTEL BOIS 4EVER')
print('=' * 60)
print(f"  Chat duration         : {total_days} days")
print(f"  From                  : {start_date.strftime('%d %B %Y')}")
print(f"  To                    : {end_date.strftime('%d %B %Y')}")
print(f"  Total messages        : {total_msg_count:,}")
print()                                           # BUG FIX 1: print('\n') → print()
print('  --- Leaderboard ---')
print()                                           # BUG FIX 1: print('\n') → print()
for sender, count in ranked_senders:
    bar = make_bar(count, highest_count)
    print(f"  {sender:<15} : {count:>5,} {bar}  {avg_words.get(sender, 0.0):>4.1f} words/msg  {media_count.get(sender, 0):>2} media  {deleted_count.get(sender, 0):>2} deleted")
                                                  # BUG FIX 2: removed emoji, fixed alignment
print()
print('  --- Average message length ---')
print()
for sender, count in ranked_senders:
    print(f"  {sender:<15} : {avg_words.get(sender, 0.0):>4.1f} words/msg")
print('=' * 60)

         GROUP OVERVIEW — HOSTEL BOIS 4EVER
  Chat duration         : 60 days
  From                  : 01 April 2024
  To                    : 30 May 2024
  Total messages        : 3,127

  --- Leaderboard ---

  Rahul           :   940 ████████████████████   2.6 words/msg   7 media   6 deleted
  Priya           :   712 ███████████████.....   5.0 words/msg   4 media   2 deleted
  Neha            :   624 █████████████.......   5.3 words/msg   8 media   3 deleted
  Aman            :   484 ██████████..........   5.0 words/msg   4 media   2 deleted
  Karan           :   345 ███████.............  57.0 words/msg   7 media   2 deleted
  Vikas           :    22 ....................   1.8 words/msg   2 media   0 deleted

  --- Average message length ---

  Rahul           :  2.6 words/msg
  Priya           :  5.0 words/msg
  Neha            :  5.3 words/msg
  Aman            :  5.0 words/msg
  Karan           : 57.0 words/msg
  Vikas           :  1.8 words/msg


## Feature 3 — Most Active Day and Hour

Find the single busiest calendar day and the busiest hour-of-day across the entire 60-day window.


In [ ]:
#Counting messages per day

messages_per_day = {}

for msg in real_messages:
    # Get just the date part like "04 May 2024"
    day = msg['Datetime'].strftime('%d %B %Y')

    if day in messages_per_day:
        messages_per_day[day] += 1
    else:
        messages_per_day[day] = 1

#Counting messages per hour ---
#grouping by hour (0 to 23)

messages_per_hour = {}

for msg in real_messages:
    #hour gives us a number like 0, 1, 2 ... 23
    hour = msg['Datetime'].hour

    if hour in messages_per_hour:
        messages_per_hour[hour] += 1
    else:
        messages_per_hour[hour] = 1

#Finding the busiest day
# We want to find the day with the highest message count

busiest_day       = None
busiest_day_count = 0

for day, count in messages_per_day.items():
    if count > busiest_day_count:
        busiest_day_count = count
        busiest_day       = day

#Finding the busiest hour

busiest_hour       = None
busiest_hour_count = 0

for hour, count in messages_per_hour.items():
    if count > busiest_hour_count:
        busiest_hour_count = count
        busiest_hour       = hour

#Average messages per hour across 60 days

avg_per_hour = {}

for hour, count in messages_per_hour.items():
    avg_per_hour[hour] = round(count / total_days, 1)

#Bar chart

def make_bar(value, max_val, bar_length=20):
    if max_val == 0:
        return '.' * bar_length
    filled = int(round((value / max_val) * bar_length))
    empty  = bar_length - filled
    return '█' * filled + '.' * empty

#results

print('=' * 55)
print('       MOST ACTIVE DAY AND HOUR')
print('=' * 55)
print(f"  Busiest Day  : {busiest_day} ({busiest_day_count} messages)")
print(f"  Busiest Hour : {busiest_hour:02d}:00 — {busiest_hour + 1:02d}:00")
print(f"  (avg {avg_per_hour[busiest_hour]} messages/day during this hour)")
print()

#Printing hourly bar chart #used AI-assistant here
#It Shows here how active the group was at each hour of the day

print('  HOURLY ACTIVITY (all 60 days combined)')
print('  ' + '-' * 45)

max_in_hour = max(messages_per_hour.values())

for hr in range(24):
    count = messages_per_hour.get(hr, 0)
    bar   = make_bar(count, max_in_hour, bar_length=25)

    # Mark the peak hour with an arrow
    if hr == busiest_hour:
        label = ' <- PEAK HOUR'
    else:
        label = ''

    print(f"  {hr:02d}:00  {bar}  {count}{label}")

print('=' * 55)

       MOST ACTIVE DAY AND HOUR
  Busiest Day  : 04 May 2024 (74 messages)
  Busiest Hour : 18:00 — 19:00
  (avg 4.1 messages/day during this hour)

  HOURLY ACTIVITY (all 60 days combined)
  ---------------------------------------------
  00:00  ██████...................  56
  01:00  ████████.................  82
  02:00  █████████................  83
  03:00  ████████.................  77
  04:00  ███████████..............  109
  05:00  ███......................  28
  06:00  ███......................  33
  07:00  ██████...................  55
  08:00  ████████████.............  117
  09:00  ███████████████..........  149
  10:00  ████████████████.........  158
  11:00  ████████████.............  113
  12:00  ███████████████████......  190
  13:00  ████████████████.........  156
  14:00  ████████████████.........  161
  15:00  █████████████............  128
  16:00  ███████████████████......  185
  17:00  █████████████████........  170
  18:00  █████████████████████████  244 <- PEAK H


## Feature 4 — Activity Heatmap (NumPy)

A **6 × 24 NumPy matrix** where rows = participants, columns = hours of day (0–23). Each cell contains the total number of messages that person sent during that hour across all 60 days.

This is the project's core NumPy moment — and the data structure that powers archetype detection (Night Owl detection lives here).


In [ ]:
#Participants #Used an AI-assistant for reference

participant_order = ['Rahul', 'Priya', 'Aman', 'Karan', 'Neha', 'Vikas']

#Maping each person to a row number
# Rahul → row 0, Priya → row 1, Aman → row 2 ... and so on

person_row = {}
for idx, name in enumerate(participant_order):
    person_row[name] = idx

#Creating the NumPy matrix
#6 rows (one per person)× 24 columns (one per hour)
#All values starting at 0

activity_matrix = np.zeros((6, 24), dtype=int)

#Filling the matrix ---
#For every message, find the sender's row and the message hour
#Then adding 1 to that cell

for msg in real_messages:
    sender = msg['Sender']
    hour   = msg['Datetime'].hour

    if sender in person_row:
        row = person_row[sender]
        col = hour
        activity_matrix[row, col] += 1

#Verifying the matrix is correct

print('=' * 55)
print('  HEATMAP VALIDATION')
print('  ' + '-' * 45)

for name in participant_order:
    row          = person_row[name]
    row_total    = activity_matrix[row].sum()
    expected     = msg_count_per_person.get(name, 0)

    if row_total == expected:
        status = 'OK'
    else:
        status = 'MISMATCH'

    print(f"  {name:<8} : {row_total} messages  (expected {expected})  {status}")

#Shading cell helper function

def shade_cell(value, personal_max):
    if personal_max == 0 or value == 0:
        return '.  '     # no activity

    ratio = value / personal_max

    if ratio < 0.25:
        return '░  '     # low activity
    elif ratio < 0.50:
        return '▒  '     # medium activity
    elif ratio < 0.75:
        return '█  '     # high activity
    else:
        return '██ '     # peak activity

#Printing the heatmap ---

print()
print('  ACTIVITY HEATMAP (messages by hour of day)')
print('  ' + '-' * 55)
print(f"  {'Person':<10}  00   03   06   09   12   15   18   21")
print('  ' + '-' * 55)

# We only show every 3rd hour (0, 3, 6, 9, 12, 15, 18, 21)
# to keep the output clean and readable
hours_to_show = [0, 3, 6, 9, 12, 15, 18, 21]

for name in participant_order:
    row          = person_row[name]
    row_data     = activity_matrix[row]
    personal_max = int(row_data.max())

    # Build the shaded row
    shaded_row = ''
    for hr in hours_to_show:
        shaded_row += shade_cell(row_data[hr], personal_max)

    # Checking if this person is a night owl
    # Night hours = 11 PM (23) and 12 AM to 4 AM (0, 1, 2, 3, 4)
    night_hours   = [23, 0, 1, 2, 3, 4]
    night_msgs    = int(np.sum(row_data[night_hours]))
    total_msgs    = int(row_data.sum())

    if total_msgs > 0:
        night_pct = round(night_msgs / total_msgs * 100, 1)
    else:
        night_pct = 0

    # Adding night owl label if more than 60% messages are at night
    if night_pct > 60:
        label = f'  <- NIGHT OWL ({night_pct}% after 11 PM)'
    else:
        label = ''

    print(f"  {name:<10}  {shaded_row}{label}")

print()
print('  Shade key:  .  = none   ░ = low   ▒ = medium   █ = high   ██ = peak')

#Some useful NumPy stats from the matrix

print()
print('  NUMPY MATRIX INSIGHTS')
print('  ' + '-' * 40)
print(f"  Matrix size              : {activity_matrix.shape}  (6 people x 24 hours)")
print(f"  Total messages in matrix : {int(activity_matrix.sum())}")

#Which hour has the most messages globally
peak_hour   = int(activity_matrix.sum(axis=0).argmax())
print(f"  Most active hour overall : {peak_hour:02d}:00")

#Which person has the most messages overall
peak_person = participant_order[int(activity_matrix.sum(axis=1).argmax())]
print(f"  Most active person       : {peak_person}")

print('=' * 55)

  HEATMAP VALIDATION
  ---------------------------------------------
  Rahul    : 940 messages  (expected 940)  OK
  Priya    : 712 messages  (expected 712)  OK
  Aman     : 484 messages  (expected 484)  OK
  Karan    : 345 messages  (expected 345)  OK
  Neha     : 624 messages  (expected 624)  OK
  Vikas    : 22 messages  (expected 22)  OK

  ACTIVITY HEATMAP (messages by hour of day)
  -------------------------------------------------------
  Person      00   03   06   09   12   15   18   21
  -------------------------------------------------------
  Rahul       ░  ░  ░  ░  █  █  ██ ██ 
  Priya       .  .  ░  ██ ██ ▒  █  █  
  Aman        █  █  .  .  .  ░  ░  ░    <- NIGHT OWL (80.4% after 11 PM)
  Karan       .  .  .  ▒  ██ █  █  ▒  
  Neha        .  .  ░  ██ █  ░  ██ ▒  
  Vikas       .  .  .  ▒  ▒  ▒  █  ▒  

  Shade key:  .  = none   ░ = low   ▒ = medium   █ = high   ██ = peak

  NUMPY MATRIX INSIGHTS
  ----------------------------------------
  Matrix size              : (6, 24)


## Feature 5 — Top Words

Extract every word from every real message, count frequency, and display the top 10 group-wide words with a proportional bar chart. We build the word counter from scratch using a plain dict — no `collections.Counter` allowed.


In [ ]:
#Defining stop words
#These are very common words that appear in every conversation
# Note: AI-assisted — used Claude to help identify stop words
#and explaining string cleaning

stop_words = {
    'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
    'it', 'was', 'that', 'this', 'my', 'me', 'you', 'we', 'he', 'she',
    'they', 'be', 'are', 'at', 'but', 'so', 'do', 'go', 'get', 'have',
    'had', 'has', 'not', 'by', 'with', 'from', 'about', 'just', 'its',
    'an', 'as', 'up', 'if', 'no', 'now', 'can', 'will', 'all', 'what',
    'your', 'out', 'there', 'when', 'been', 'one', 'who', 'how', 'more',
    'some', 'than', 'him', 'then', 'them', 'also', 'would', 'could',
    'should', 'dont', 'im', 'ive', 'wont', 'didnt', 'am', 'his', 'her',
    'our', 'which', 'were', 'their', 'said', 'after', 'even', 'did',
    'very', 'too', 'back', 'into', 'over', 'again', 'came', 'went',
    'told', 'come', 'like', 'got', 'see', 'well', 'really', 'know',
    'think', 'want', 'need', 'day', 'time', 'thing', 'guys', 'everyone',
    'someone', 'anyone', 'because', 'actually', 'started', 'telling',
    'whole', 'still', 'every', 'long', 'little',
}

#Word cleaning function
#Removes punctuation from both sides and converts to lowercase
#Example: "Bhai!" → "bhai"  |  "YAAR," → "yaar"

def clean_word(raw_word):
    word = raw_word.strip(string.punctuation)
    word = word.lower()
    return word

#Counting word frequency across all messages
#Two dictionaries:
#global_word_freq    → counts every word across the whole group
#per_person_word_freq → counts words separately for each person

global_word_freq     = {}
per_person_word_freq = {}

#Set up an empty dict for each person
for name in participant_order:
    per_person_word_freq[name] = {}

#Go through every real message
for msg in real_messages:
    sender   = msg['Sender']
    all_words = msg['Message_Text'].split()

    for raw_word in all_words:
        word = clean_word(raw_word)

        #Skip if empty after cleaning
        if len(word) == 0:
            continue

        #Skip very short words (single characters)
        if len(word) < 2:
            continue

        #Skip stop words — not useful for analysis
        if word in stop_words:
            continue

        #Skip pure numbers like "2", "10", "2024"
        if word.isdigit():
            continue

        #Add to global count
        if word in global_word_freq:
            global_word_freq[word] += 1
        else:
            global_word_freq[word] = 1

        #Add to that person's personal count
        if sender in per_person_word_freq:
            if word in per_person_word_freq[sender]:
                per_person_word_freq[sender][word] += 1
            else:
                per_person_word_freq[sender][word] = 1

#Finding the top 10 most used words overall
#sorted() with reverse=True gives highest count first
#[:10] keeps only the first 10

all_words_sorted = sorted(global_word_freq.items(), key=lambda x: x[1], reverse=True)
top10_words      = all_words_sorted[:10]

#The highest count is used to scale the bar chart
if len(top10_words) > 0:
    max_word_count = top10_words[0][1]
else:
    max_word_count = 1

#Find top 5 words per person

top5_per_person = {}

for name in participant_order:
    person_words        = per_person_word_freq[name]
    person_words_sorted = sorted(person_words.items(), key=lambda x: x[1], reverse=True)
    top5_per_person[name] = person_words_sorted[:5]

#results

print('=' * 55)
print("  THIS GROUP'S FAVOURITE WORDS")
print('  ' + '-' * 45)

for word, count in top10_words:
    bar = make_bar(count, max_word_count)
    print(f"  {word:<14}  {bar}  {count}")

print()
print('  TOP 5 WORDS PER PERSON')
print('  ' + '-' * 45)

for name in participant_order:
    top5      = top5_per_person[name]
    word_list = ''

    for w, c in top5:
        word_list += f'{w}({c})  '

    print(f"  {name:<10} ->  {word_list}")

print('=' * 55)

  THIS GROUP'S FAVOURITE WORDS
  ---------------------------------------------
  hai             ████████████████████  268
  today           ███████████████████.  257
  bhai            ████████████........  160
  scene           ███████████.........  145
  entire          ███████████.........  145
  please          ███████████.........  141
  yaar            ██████████..........  139
  kya             ██████████..........  133
  everything      █████████...........  121
  sleep           ████████............  112

  TOP 5 WORDS PER PERSON
  ---------------------------------------------
  Rahul      ->  hai(263)  bhai(159)  scene(144)  kya(133)  yaar(105)  
  Priya      ->  please(141)  aman(93)  okay(80)  take(72)  eat(60)  
  Aman       ->  sleep(71)  i'm(56)  wonder(43)  night(41)  can't(34)  
  Karan      ->  today(171)  entire(145)  three(97)  used(95)  everything(92)  
  Neha       ->  cant(58)  ok(52)  today(48)  way(48)  wait(48)  
  Vikas      ->  hai(5)  haha(4)  sorry(3)  bus


## Feature 6 — Response Speed & Silent Streaks

Two behavioural metrics that expose the group's fastest and slowest communicators:

1. **Average response time** — for each person, the average time gap between the last message from someone else and their next reply
2. **Longest silent streak** — the longest run of consecutive days where a person sent zero messages


In [ ]:
#Seting up a list to collect response gaps per person
#For each person we collect all their response times in seconds

response_gaps = {}
for name in participant_order:
    response_gaps[name] = []

#Calculate response time for each message
#Logic:
#For every message, look BACKWARDS to find the last message
#sent by someone DIFFERENT
#The time gap between that message and this one = response time
#We ignore gaps over 24 hours (86400 seconds) because that is
#not really a response — the group was just inactive

#We start from index 1 because index 0 has no previous message
for current_idx in range(1, len(real_messages)):
    current_msg    = real_messages[current_idx]
    current_sender = current_msg['Sender']

    #Walk backwards through previous messages
    for prev_idx in range(current_idx - 1, -1, -1):
        prev_msg    = real_messages[prev_idx]
        prev_sender = prev_msg['Sender']

        #We only care about messages from a DIFFERENT person
        if prev_sender != current_sender:
            gap = (current_msg['Datetime'] - prev_msg['Datetime']).total_seconds()

            #Only count if gap is between 0 and 24 hours
            if gap > 0 and gap < 86400:
                response_gaps[current_sender].append(gap)

            #Stop looking backwards — we found the most recent other person
            break

#Calculate average response time per person

avg_response = {}

for name in participant_order:
    gaps = response_gaps[name]

    if len(gaps) > 0:
        total_gap        = 0
        for g in gaps:
            total_gap   += g
        avg_response[name] = total_gap / len(gaps)
    else:
        avg_response[name] = None    # no data for this person

#Helper function to show time in readable format
#Converts raw seconds into "35 sec" or "4.2 min" or "6.8 hrs"

def format_time(seconds):
    if seconds is None:
        return 'no data'
    elif seconds < 120:
        return f"{seconds:.0f} sec"
    elif seconds < 3600:
        return f"{seconds / 60:.1f} min"
    else:
        return f"{seconds / 3600:.1f} hrs"

#Find fastest and slowest replier

fastest_name  = None
fastest_time  = float('inf')
slowest_name  = None
slowest_time  = 0

for name in participant_order:
    t = avg_response[name]
    if t is None:
        continue
    if t < fastest_time:
        fastest_time = t
        fastest_name = name
    if t > slowest_time:
        slowest_time = t
        slowest_name = name

#Print response time results

print('=' * 55)
print('  RESPONSE PATTERNS')
print('  ' + '-' * 45)
print(f"  Fastest replier : {fastest_name}  (avg {format_time(fastest_time)})")
print(f"  Slowest replier : {slowest_name}  (avg {format_time(slowest_time)})")
print()
print('  Response time per person:')
print('  ' + '-' * 35)

for name in participant_order:
    t = avg_response[name]
    print(f"  {name:<10} : {format_time(t)}")

# SILENT STREAK CALCULATION


#Finding the longest silent streak per person ---
#A silent streak = consecutive days where the person sent 0 messages
#We go through all 60 days one by one and check if the person was active or not

streak_results = []

for name in participant_order:

    #Collect all dates this person sent a message
    active_days = set()
    for msg in real_messages:
        if msg['Sender'] == name:
            active_days.add(msg['Datetime'].date())

    #Now walk through every day in the 60 day window
    longest_streak       = 0
    longest_streak_start = None
    current_streak       = 0
    current_start        = None

    for day_num in range(total_days):
        #Figure out what date this day number is
        this_date = start_date + timedelta(days=day_num)

        if this_date not in active_days:
            #Person was silent this day
            if current_streak == 0:
                current_start = this_date   # mark when silence started
            current_streak += 1

            #Update longest if this streak beats previous best
            if current_streak > longest_streak:
                longest_streak       = current_streak
                longest_streak_start = current_start
        else:
            #Person was active — reset the streak counter
            current_streak = 0

    streak_results.append((name, longest_streak, longest_streak_start))

# Sort from longest streak to shortest
streak_results.sort(key=lambda x: x[1], reverse=True)

#results

print()
print('  LONGEST SILENT STREAKS')
print('  (consecutive days with zero messages)')
print('  ' + '-' * 45)

for name, streak_len, streak_start in streak_results:
    if streak_len == 0:
        print(f"  {name:<10} : never went silent (0 days)")
    else:
        #Calculate when the streak ended
        streak_end    = streak_start + timedelta(days=streak_len - 1)
        date_range    = f"({streak_start.strftime('%d %b')} to {streak_end.strftime('%d %b')})"
        print(f"  {name:<10} : {streak_len} days  {date_range}")

print('=' * 55)

  RESPONSE PATTERNS
  ---------------------------------------------
  Fastest replier : Vikas  (avg 35.3 min)
  Slowest replier : Aman  (avg 2.7 hrs)

  Response time per person:
  -----------------------------------
  Rahul      : 40.3 min
  Priya      : 1.2 hrs
  Aman       : 2.7 hrs
  Karan      : 45.2 min
  Neha       : 50.0 min
  Vikas      : 35.3 min

  LONGEST SILENT STREAKS
  (consecutive days with zero messages)
  ---------------------------------------------
  Vikas      : 11 days  (23 Apr to 03 May)
  Rahul      : never went silent (0 days)
  Priya      : never went silent (0 days)
  Aman       : never went silent (0 days)
  Karan      : never went silent (0 days)
  Neha       : never went silent (0 days)


---
## Feature 7 — Personality Archetype Detection

Each of the 8 archetypes has a detection function that takes a person's data and returns a numeric score. Every person is then assigned the archetype where they score highest — exclusive assignment, so no two people can share the same top archetype.

**The 8 archetypes:**

| Archetype | Detection Rule |
|---|---|
| THE SPAMMER | Avg consecutive message burst > 3 |
| THE GROUP MOM | Highest count of caring keywords |
| THE NIGHT OWL | > 60% messages between 23:00–04:59 |
| THE STORYTELLER | Avg words per message > 30 |
| THE DRAMA QUEEN | > 30% messages are all-caps or have 2+ exclamation marks |
| THE GHOST | Silent on > 60% of days |
| THE COMEDIAN | Highest % of lol/lmao/haha etc. |
| THE QUESTION MASTER | > 25% messages end with '?' |
| THE INSOMNIAC PHILOSOPHER *(bonus)* | Uses 'life', 'meaning', 'time', 'world', 'think' after midnight |


In [ ]:
#Detection function: THE SPAMMER

def score_spammer(person_name, all_msgs):
    bursts = []
    current_burst = 0
    for msg in all_msgs:
        if msg['Sender'] == person_name:
            current_burst += 1
        else:
            if current_burst > 0:
                bursts.append(current_burst)
            current_burst = 0
    if current_burst > 0:
        bursts.append(current_burst)
    return round(sum(bursts) / len(bursts), 2) if bursts else 0.0

#---------------------------------------------------------------------
#Detection function: THE GROUP MOM
#Counts how many messages contain caring/nurturing keywords.

CARING_KEYWORDS = {
    'okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please',
    'reminder', 'drink water', "don't forget", 'help', 'need', 'worried',
    'feeling', 'better', 'rest', 'careful', 'alright', 'fine', 'doing okay'
}

def score_group_mom(person_name, all_msgs):
    person_msgs = [m['Message_Text'].lower() for m in all_msgs if m['Sender'] == person_name]
    caring_hits = 0
    for msg_text in person_msgs:
        for keyword in CARING_KEYWORDS:
            if keyword in msg_text:
                caring_hits += 1
                break   #count each message once even if it has multiple keywords
    return caring_hits

# ------------------------------------------------------------------
#Detection function: THE NIGHT OWL
#Uses the NumPy matrix — sum the night hour columns (23,0,1,2,3,4

NIGHT_HOURS = [23, 0, 1, 2, 3, 4]

def score_night_owl(person_name, matrix, row_index):
    row = matrix[row_index[person_name]]
    row_total = int(row.sum())
    if row_total == 0:
        return 0.0
    night_msgs = int(np.sum(row[NIGHT_HOURS]))
    return round(night_msgs / row_total * 100, 2)

# ------------------------------------------------------------------
#Detection function: THE STORYTELLER

def score_storyteller(person_name, all_msgs):
    msgs = [m for m in all_msgs if m['Sender'] == person_name]
    if not msgs:
        return 0.0
    total_words = sum(len(m['Message_Text'].split()) for m in msgs)
    return round(total_words / len(msgs), 2)

# ------------------------------------------------------------------
#Detection function: THE DRAMA QUEEN
#Counts messages that are:
#(a) entirely alphabetic-uppercase (using isupper()), OR
#(b) contain 2 or more exclamation marks
#Short messages under 3 alphabetic characters are excluded.

def score_drama_queen(person_name, all_msgs):
    msgs = [m['Message_Text'] for m in all_msgs if m['Sender'] == person_name]
    drama_count = 0
    eligible    = 0
    for msg in msgs:
        alpha_chars = ''.join(c for c in msg if c.isalpha())
        if len(alpha_chars) < 3:
            continue
        eligible += 1
        if alpha_chars.isupper() or msg.count('!') >= 2:
            drama_count += 1
    return round(drama_count / eligible * 100, 2) if eligible > 0 else 0.0

# ------------------------------------------------------------------
#Detection function: THE GHOST
#Percentage of days in the 60-day window where the person sent

def score_ghost(person_name, all_msgs, total_days_count, start):
    active_dates = set(
        m['Datetime'].date() for m in all_msgs if m['Sender'] == person_name
    )
    silent_days = total_days_count - len(active_dates)
    return round(silent_days / total_days_count * 100, 2)

# ------------------------------------------------------------------
#Detection function: THE COMEDIAN
#Percentage of messages containing any laugh word.

LAUGH_WORDS = {'lol', 'lmao', 'haha', 'rofl', 'lmfao', 'hehe', 'hahaha'}

def score_comedian(person_name, all_msgs):
    msgs = [m['Message_Text'].lower() for m in all_msgs if m['Sender'] == person_name]
    if not msgs:
        return 0.0
    laugh_count = sum(
        1 for msg in msgs
        if any(lw in msg.split() or lw in msg for lw in LAUGH_WORDS)
    )
    return round(laugh_count / len(msgs) * 100, 2)

# ------------------------------------------------------------------
#Detection function: THE QUESTION MASTER
#Percentage of messages that end with a '?' character.

def score_question_master(person_name, all_msgs):
    msgs = [m['Message_Text'].strip() for m in all_msgs if m['Sender'] == person_name]
    if not msgs:
        return 0.0
    question_count = sum(1 for msg in msgs if msg.endswith('?'))
    return round(question_count / len(msgs) * 100, 2)

# ------------------------------------------------------------------
#BONUS Archetype: THE INSOMNIAC PHILOSOPHER
#Uses introspective words ('life', 'meaning', 'world', 'think',
# 'time', 'wonder', 'feel') specifically after midnight (00:00–04:59)
#This archetype captures the 'deep 3 AM conversations' pattern common in indian college hostel groups

PHILOSOPHY_WORDS = {'life', 'meaning', 'world', 'think', 'time', 'wonder',
                    'feel', 'purpose', 'exist', 'reality', 'truly', 'always',
                    'never', 'everything', 'nothing', 'maybe', 'perhaps'}

def score_insomniac_philosopher(person_name, all_msgs):
    late_msgs = [
        m['Message_Text'].lower() for m in all_msgs
        if m['Sender'] == person_name and m['Datetime'].hour in [0, 1, 2, 3, 4]
    ]
    if not late_msgs:
        return 0.0
    phil_count = sum(
        1 for msg in late_msgs
        if any(pw in msg.split() for pw in PHILOSOPHY_WORDS)
    )
    return round(phil_count / len(late_msgs) * 100, 2)

# ------------------------------------------------------------------
# SCORE MATRIX: compute every person's score for every archetype.
# Then assign each archetype exclusively to the person with the highest score fot that archetype

archetype_names = [
    'THE SPAMMER',
    'THE GROUP MOM',
    'THE NIGHT OWL',
    'THE STORYTELLER',
    'THE DRAMA QUEEN',
    'THE GHOST',
    'THE COMEDIAN',
    'THE QUESTION MASTER',
    'THE INSOMNIAC PHILOSOPHER',
]

#Building the full score matrix as a dict of dicts
score_matrix = {}
for person in participant_order:
    score_matrix[person] = {
        'THE SPAMMER':               score_spammer(person, real_messages),
        'THE GROUP MOM':             score_group_mom(person, real_messages),
        'THE NIGHT OWL':             score_night_owl(person, activity_matrix, person_row),
        'THE STORYTELLER':           score_storyteller(person, real_messages),
        'THE DRAMA QUEEN':           score_drama_queen(person, real_messages),
        'THE GHOST':                 score_ghost(person, real_messages, total_days, start_date),
        'THE COMEDIAN':              score_comedian(person, real_messages),
        'THE QUESTION MASTER':       score_question_master(person, real_messages),
        'THE INSOMNIAC PHILOSOPHER': score_insomniac_philosopher(person, real_messages),
    }

#Print raw scores for transparency
print("  RAW ARCHETYPE SCORES")
print(f"  {'':30}" + '  '.join(f'{p[:5]:>6}' for p in participant_order))
print('  ' + '-' * 68)
for archetype in archetype_names:
    scores_row = '  '.join(f"{score_matrix[p][archetype]:>6.1f}" for p in participant_order)
    print(f"  {archetype:<30} {scores_row}")

#Greedy exclusive assignment
#Building all (archetype, person, score) triples
all_triples = []
for person in participant_order:
    for archetype, score in score_matrix[person].items():
        all_triples.append((archetype, person, score))

#Sorting by score descending
all_triples.sort(key=lambda t: -t[2])

assigned_archetypes  = {}   # person → archetype
claimed_archetypes   = set()  # archetypes already assigned

for archetype, person, score in all_triples:
    if person not in assigned_archetypes and archetype not in claimed_archetypes:
        assigned_archetypes[person]  = (archetype, score)
        claimed_archetypes.add(archetype)

print()
print("  FINAL ARCHETYPE ASSIGNMENTS")
print('  ' + '-' * 50)
for person in participant_order:
    archetype, score = assigned_archetypes.get(person, ('UNKNOWN', 0))
    print(f"  {person:<8} → {archetype:<30} (score: {score:.1f})")


  RAW ARCHETYPE SCORES
                                 Rahul   Priya    Aman   Karan    Neha   Vikas
  --------------------------------------------------------------------
  THE SPAMMER                       4.5     1.7     2.8     1.2     2.5     1.1
  THE GROUP MOM                     0.0   509.0    97.0   123.0    66.0     0.0
  THE NIGHT OWL                    13.5     1.3    80.4     2.0     4.8     9.1
  THE STORYTELLER                   2.5     5.0     5.0    57.0     5.3     1.8
  THE DRAMA QUEEN                   0.0     0.0     0.0     0.0    63.3     0.0
  THE GHOST                         0.0     0.0     0.0     0.0     0.0    73.3
  THE COMEDIAN                      3.2     0.0     0.0     0.0     0.0    18.2
  THE QUESTION MASTER               3.8    29.5     6.6     0.0     6.2     0.0
  THE INSOMNIAC PHILOSOPHER         0.0     0.0    22.8     0.0     0.0     0.0

  FINAL ARCHETYPE ASSIGNMENTS
  --------------------------------------------------
  Rahul    → THE SPAMME


## Feature 8 — The Final Report

Everything assembled into one clean, formatted, screenshot-worthy terminal report. This is the cell you run last, the one you screenshot for LinkedIn.


In [ ]:
#FINAL REPORT
#everything printed together

#Helper: Draw a divider line
DIVIDER      = '=' * 60
THIN_DIVIDER = '-' * 60

#Helper: Center any text inside the report
def center_text(text):
    return text.center(60)

#Helper: Draw a bar chart for any value
def draw_bar(value, max_val, bar_length=20):
    if max_val == 0:
        return '.' * bar_length
    filled = int(round((value / max_val) * bar_length))
    empty  = bar_length - filled
    return '█' * filled + '.' * empty

# ---------------------------------------------------------------------------------------------------------------------
#REPORT

print(DIVIDER)
print(center_text('GROUPDNA REPORT'))
print(center_text('"Hostel Bois 4ever"'))
print(center_text(f'{total_days} days  |  {len(real_messages):,} messages  |  {len(participant_order)} members'))
print(DIVIDER)
print()

#1: Basic Info----------------------------------------------------------------------------------------------------------

print(f"  Period       : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')}")
print(f"  Busiest Day  : {busiest_day} ({messages_per_day[busiest_day]} messages)")
print(f"  Busiest Hour : {busiest_hour:02d}:00 to {busiest_hour + 1:02d}:00")
print()

#2: Messages Per Person-----------------------------------------------------------------------

print('  MESSAGES PER PERSON')
print('  ' + THIN_DIVIDER)

highest = max(msg_count_per_person.values())

for person, count in ranked_senders:
    bar = draw_bar(count, highest)
    pct = round(count / total_msg_count * 100, 1)
    print(f"  {person:<10} {bar}  {count:>4} ({pct}%)")

print()

#3: Activity Heatmap------------------------------------------------------

print('  ACTIVITY HEATMAP (hour of day)')
print(f"  {'Person':<10}  00  03  06  09  12  15  18  21")
print('  ' + THIN_DIVIDER)

hours_to_show = [0, 3, 6, 9, 12, 15, 18, 21]

for name in participant_order:
    row          = person_row[name]
    row_data     = activity_matrix[row]
    personal_max = int(row_data.max())

    #Build shaded row
    shaded_row = ''
    for hr in hours_to_show:
        shaded_row += shade_cell(row_data[hr], personal_max)

    #Check night owl
    night_hours = [23, 0, 1, 2, 3, 4]
    night_msgs  = int(np.sum(row_data[night_hours]))
    total_p     = int(row_data.sum())

    if total_p > 0:
        night_pct = round(night_msgs / total_p * 100, 1)
    else:
        night_pct = 0

    if night_pct > 60:
        tag = '  <- NIGHT OWL'
    else:
        tag = ''

    print(f"  {name:<10}  {shaded_row}{tag}")

print()

#4: Top Words-----------------------------------------------------------------------

print("  THIS GROUP'S FAVOURITE WORDS")
print('  ' + THIN_DIVIDER)

max_word_count = top10_words[0][1]

for word, count in top10_words:
    bar = draw_bar(count, max_word_count)
    print(f"  {word:<15} {bar}  {count}")

print()

#5: Response Patterns----------------------------------------------------------------------

print('  RESPONSE PATTERNS')
print('  ' + THIN_DIVIDER)
print(f"  Fastest Replier : {fastest_name:<10} (avg {format_time(fastest_time)})")
print(f"  Slowest Replier : {slowest_name:<10} (avg {format_time(slowest_time)})")
print()

#6: Silent Streaks---------------------------------------------------------------------

print('  LONGEST SILENT STREAKS')
print('  ' + THIN_DIVIDER)

for name, streak_len, streak_start in streak_results:
    if streak_len == 0:
        print(f"  {name:<10} : never went silent")
    else:
        streak_end = streak_start + timedelta(days=streak_len - 1)
        date_range = f"({streak_start.strftime('%d %b')} to {streak_end.strftime('%d %b')})"
        print(f"  {name:<10} : {streak_len} days  {date_range}")

print()

#7: Personality Archetypes --------------------------------------------------------------------------

print('  PERSONALITY ARCHETYPES')
print('  ' + THIN_DIVIDER)

#Detail line for each archetype — describes WHY they got it
def get_archetype_detail(person, archetype):
    if archetype == 'THE SPAMMER':
        return f"avg {score_matrix[person]['THE SPAMMER']:.1f} msgs in a row"
    elif archetype == 'THE GROUP MOM':
        return f"caring keyword score: {score_matrix[person]['THE GROUP MOM']:.0f}"
    elif archetype == 'THE NIGHT OWL':
        return f"{score_matrix[person]['THE NIGHT OWL']:.1f}% msgs after 11 PM"
    elif archetype == 'THE STORYTELLER':
        return f"avg {score_matrix[person]['THE STORYTELLER']:.1f} words per msg"
    elif archetype == 'THE DRAMA QUEEN':
        return f"{score_matrix[person]['THE DRAMA QUEEN']:.1f}% all caps messages"
    elif archetype == 'THE GHOST':
        silent_days = int(score_matrix[person]['THE GHOST'] * total_days / 100)
        return f"silent on {silent_days} of {total_days} days"
    elif archetype == 'THE COMEDIAN':
        return f"{score_matrix[person]['THE COMEDIAN']:.1f}% messages have laugh words"
    elif archetype == 'THE QUESTION MASTER':
        return f"{score_matrix[person]['THE QUESTION MASTER']:.1f}% messages are questions"
    elif archetype == 'THE INSOMNIAC PHILOSOPHER':
        return f"{score_matrix[person]['THE INSOMNIAC PHILOSOPHER']:.1f}% late night msgs are deep"
    else:
        return ''

for name in participant_order:
    archetype, score = assigned_archetypes.get(name, ('UNKNOWN', 0))
    detail           = get_archetype_detail(name, archetype)
    print(f"  {name:<10} ->  {archetype:<30}  ({detail})")

# ----------------------------------------------------------------
#END OF REPORT


print()
print(DIVIDER)
print(center_text('Generated by GroupDNA'))
print(center_text('Built with Python + NumPy  |  No pandas. No shortcuts.'))
print(DIVIDER)

                      GROUPDNA REPORT                       
                    "Hostel Bois 4ever"                     
          60 days  |  3,127 messages  |  6 members          

  Period       : 01 April 2024 to 30 May 2024
  Busiest Day  : 04 May 2024 (74 messages)
  Busiest Hour : 18:00 to 19:00

  MESSAGES PER PERSON
  ------------------------------------------------------------
  Rahul      ████████████████████   940 (30.1%)
  Priya      ███████████████.....   712 (22.8%)
  Neha       █████████████.......   624 (20.0%)
  Aman       ██████████..........   484 (15.5%)
  Karan      ███████.............   345 (11.0%)
  Vikas      ....................    22 (0.7%)

  ACTIVITY HEATMAP (hour of day)
  Person      00  03  06  09  12  15  18  21
  ------------------------------------------------------------
  Rahul       ░  ░  ░  ░  █  █  ██ ██ 
  Priya       .  .  ░  ██ ██ ▒  █  █  
  Aman        █  █  .  .  .  ░  ░  ░    <- NIGHT OWL
  Karan       .  .  .  ▒  ██ █  █  ▒  
  Neha    

---
## Reflection

- ### **What was the hardest part?**

Honestly, the hardest part was not any single feature —
it was the debugging that came with every single one of
them. Each feature looked straightforward when I read it,
but the moment I started writing the code, small errors
kept showing up. Wrong key names, indentation issues,
using `line['key']` instead of `Summary['key']` — these
small mistakes took time to spot and fix. But every time
I debugged one, I understood the code a little better.

he archetype detection (Feature 7) was the most
challenging feature overall. On paper it sounds simple —
score each person, assign the highest archetype. But when
I actually sat down to build it, I realised the exclusive
assignment logic was tricky. You cannot just give everyone
the archetype they score highest on, because two people
might both score highest on the same archetype. The greedy
assignment — sort all scores globally, then assign one by
one — was not something that came to me immediately. I had
to think it through carefully before the logic clicked.




- ### **What would I do differently?**

I would plan my variable names before writing a single
line of code. A lot of my debugging time was wasted
because I used different names in different features —
`msg['sender']` in one place and `msg['Sender']` in
another. If I had locked down a naming convention on Day 1
and stuck to it, I would have saved hours of NameErrors
and KeyErrors across the project.

I would also build the parser more carefully first and
print the first few messages immediately to verify it
worked — instead of finding out three features later that
something was off from the beginning.

- ### **What archetype did I get on my own chat?**

I ran GroupDNA on my own WhatsApp group.

My archetype: **THE GHOST** 👻

Silent on more days than I was active.
The data does not lie, even when you wish it would.

Apparently I am present in the group the same way
I am present in morning lectures — technically,
but not really.

AI Assistance Disclosure

I used Claude (AI assistant) as a learning aid throughout
this project. Specifically, I used it for:

- **Debugging** — when I got NameErrors, KeyErrors, and
  indentation mistakes, I pasted my code and asked Claude
  to explain what was wrong and why. This helped me
  understand my mistakes rather than just fixing them
  blindly.

- **Understanding concepts** — string methods, how
  functions work, how constraints like `split(' - ', 1)`
  behave, and why `maxsplit=1` matters. Claude explained
  these in simple terms which made the logic click faster.

- **Stop words list** — I used Claude to help identify
  which common words to filter out in Feature 5 so the
  top words reflect the group's actual vocabulary.

- **Calculation logic** — for response time gaps and the
  silent streak counter, I used Claude to understand the
  logic before writing it in my own style.

The final code structure, variable names, and overall
notebook are written by me in my own style. Claude acted
as a teacher that pointed me in the right direction